In [ ]:
# ==============================================================================
# PROYECTO APLICADO: PREPARACIÓN DE DATOS, ENTRENAMIENTO Y REGISTRO CON MLFLOW
# Dataset: Pima Indians Diabetes (Clasificación Binaria)
# ==============================================================================

# ------------------------------------------------------------------------------
# APARTADO 1: INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN DEL ENTORNO
# ------------------------------------------------------------------------------
# Se instalan las librerías necesarias y se crean los directorios según la
# estructura solicitada en las instrucciones del proyecto.

!pip install mlflow scikit-learn pandas numpy matplotlib seaborn -q

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

# Creación de la estructura de carpetas requerida en el repositorio
os.makedirs('datos/datos_ini', exist_ok=True)
os.makedirs('datos/datos_limp', exist_ok=True)
print("[INFO] Estructura de directorios creada correctamente.")


# ------------------------------------------------------------------------------
# APARTADO 2: CARGA, LIMPIEZA Y VERSIONADO DE DATOS (TEMA 11)
# ------------------------------------------------------------------------------
# Se descarga el dataset original de Diabetes, se guarda en 'datos_ini', se
# tratan los ceros atípicos (valores nulos implícitos en variables médicas) y se
# exporta el conjunto limpio a 'datos_limp'.

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Carga y respaldo del dataset original
df_original = pd.read_csv(url, names=column_names)
df_original.to_csv('datos/datos_ini/diabetes_original.csv', index=False)

# Limpieza: Remplazo de ceros biológicamente imposibles por NaN y luego por la mediana
df = df_original.copy()
cols_con_ceros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_con_ceros:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

# Eliminación de posibles duplicados
df = df.drop_duplicates()

# Guardado del dataset procesado y limpio
df.to_csv('datos/datos_limp/diabetes_limpio.csv', index=False)
print("[INFO] Dataset procesado y guardado en 'datos/datos_limp/diabetes_limpio.csv'.")


# ------------------------------------------------------------------------------
# APARTADO 3: ANÁLISIS EXPLORATORIO DE DATOS (EDA Y VISUALIZACIONES)
# ------------------------------------------------------------------------------
# Visualizaciones relevantes para identificar la distribución de las variables y
# posibles correlaciones antes del modelado.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Visualización 1: Distribución de la Glucosa
sns.histplot(df['Glucose'], kde=True, ax=axes[0], color='teal')
axes[0].set_title('Distribución de Glucosa (Datos Limpios)')

# Visualización 2: Matriz de Correlación
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='Blues', ax=axes[1])
axes[1].set_title('Matriz de Correlación entre Variables')

plt.tight_layout()
plt.show()


# ------------------------------------------------------------------------------
# APARTADO 4: PREPARACIÓN Y DIVISIÓN DEL DATASET (TRAIN / TEST)
# ------------------------------------------------------------------------------
# División estratificada (80% entrenamiento, 20% prueba) con semilla fija (random_state=42)
# para asegurar reproducibilidad[cite: 1]. Aplicación de escalado de características.

X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado estándar (requerido para modelos basados en gradientes/distancias como Regresión Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"[INFO] Tamaño conjunto entrenamiento: {X_train.shape[0]} muestras")
print(f"[INFO] Tamaño conjunto de prueba: {X_test.shape[0]} muestras")


# ------------------------------------------------------------------------------
# APARTADO 5: ENTRENAMIENTO, BÚSQUEDA EN MALLA (GRID SEARCH) Y MLFLOW (TEMA 12)
# ------------------------------------------------------------------------------
# Entrenamiento de dos modelos (Regresión Logística vs. Random Forest), optimización
# de hiperparámetros mediante Grid Search con validación cruzada (k-fold=5) y
# registro completo de parámetros, métricas y artefactos en MLflow[cite: 1, 5].

mlflow.set_experiment("Proyecto_Diabetes_MLOps")

# --- MODELO 1: REGRESIÓN LOGÍSTICA ---
with mlflow.start_run(run_name="Regresion_Logistica"):
    # Definición de la malla de hiperparámetros
    param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100]}
    
    # Búsqueda en malla con validación cruzada de 5 pliegues (5-fold CV)[cite: 1]
    grid_lr = GridSearchCV(LogisticRegression(max_iter=1000), param_grid_lr, cv=5, scoring='accuracy')
    grid_lr.fit(X_train_scaled, y_train)
    
    # Predicciones con el mejor modelo encontrado
    best_lr = grid_lr.best_estimator_
    y_pred_lr = best_lr.predict(X_test_scaled)
    y_prob_lr = best_lr.predict_proba(X_test_scaled)[:, 1]
    
    # Registro de hiperparámetros y métricas en MLflow[cite: 1]
    mlflow.log_params(grid_lr.best_params_)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_lr))
    mlflow.log_metric("precision", precision_score(y_test, y_pred_lr))
    mlflow.log_metric("recall", recall_score(y_test, y_pred_lr))
    mlflow.log_metric("f1_score", f1_score(y_test, y_pred_lr))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_prob_lr))
    
    # Registro del artefacto del modelo (actualizado para MLflow moderno)
    mlflow.sklearn.log_model(
        sk_model=best_lr, 
        name="modelo_regresion_logistica", 
        serialization_format="cloudpickle"
    )
    print("[MLflow] Regresión Logística entrenada y registrada exitosamente.")

# --- MODELO 2: RANDOM FOREST ---
with mlflow.start_run(run_name="Random_Forest"):
    # Definición de la malla de hiperparámetros
    param_grid_rf = {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10]}
    
    # Búsqueda en malla con validación cruzada (5-fold CV)[cite: 1]
    grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy')
    grid_rf.fit(X_train, y_train)
    
    # Predicciones con el mejor modelo encontrado
    best_rf = grid_rf.best_estimator_
    y_pred_rf = best_rf.predict(X_test)
    y_prob_rf = best_rf.predict_proba(X_test)[:, 1]
    
    # Registro de hiperparámetros y métricas en MLflow[cite: 1]
    mlflow.log_params(grid_rf.best_params_)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_rf))
    mlflow.log_metric("precision", precision_score(y_test, y_pred_rf))
    mlflow.log_metric("recall", recall_score(y_test, y_pred_rf))
    mlflow.log_metric("f1_score", f1_score(y_test, y_pred_rf))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_prob_rf))
    
    # Registro del artefacto del modelo (actualizado para MLflow moderno)
    mlflow.sklearn.log_model(
        sk_model=best_rf, 
        name="modelo_random_forest", 
        serialization_format="cloudpickle"
    )
    print("[MLflow] Random Forest entrenado y registrado exitosamente.")


# ------------------------------------------------------------------------------
# APARTADO 6: COMPARATIVA DE RESULTADOS Y EVALUACIÓN VISUAL
# ------------------------------------------------------------------------------
# Generación de la matriz de confusión y la curva ROC para la comparación de ambos modelos.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matriz de Confusión - Random Forest
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Diabetes', 'Diabetes'])
disp.plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de Confusión - Random Forest')

# Curva ROC Comparativa
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

axes[1].plot(fpr_lr, tpr_lr, label=f'Regresión Logística (AUC = {roc_auc_score(y_test, y_prob_lr):.2f})')
axes[1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.2f})')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC Comparativa')
axes[1].legend()

plt.tight_layout()
plt.show()